In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from math import sqrt
from tqdm.auto import tqdm # Importar tqdm
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
caminho_val    = "/content/drive/MyDrive/Projeto Aplicado 4/dataset_val_2021_2022.csv"
caminho_treino = "/content/drive/MyDrive/Projeto Aplicado 4/dataset_train_2014_2020.csv"

In [ ]:
# Hiperparâmetros de série
HISTORY_WINDOW = 3   # anos de histórico usados como entrada
FORECAST_H     = 1   # horizonte de previsão (1 ano à frente)

# Chaves de definição para cada série temporal
GROUP_KEYS = ["CO_IES", "CO_CINE_ROTULO_id", "Modalidade"]

# Colunas contínuas e categóricas
CONT_COLS = [
    "QT_MAT", "QT_ING", "QT_ING_FEM", "QT_ING_MASC",
    "QT_SIT_TRANCADA", "QT_SIT_DESVINCULADO", "QT_SIT_TRANSFERIDO", "QT_SIT_FALECIDO",
    "QT_ING_PROCESCPUBLICA", "QT_ING_PROCESCPRIVADA"
]
CAT_COLS = [
    "Modalidade", "Rede", "CO_REGIAO", "CO_UF", "CO_MUNICIPIO", "CO_CINE_ROTULO_id"
]

#  Leitura dos dois arquivos (treino e validação)
df_train_raw = pd.read_csv(caminho_treino, low_memory=False)
df_val_raw   = pd.read_csv(caminho_val,    low_memory=False)

# Garantir tipo numérico do ano
for dfx in (df_train_raw, df_val_raw):
    dfx["NU_ANO_CENSO"] = pd.to_numeric(dfx["NU_ANO_CENSO"], errors="coerce")

In [ ]:
# Criar métrica de evasão e alvo TAXA_EVASAO
def add_evasao_cols(dfx: pd.DataFrame) -> pd.DataFrame:
    dfx = dfx.copy()
    dfx["QT_EVASAO_TOTAL"] = (
        dfx["QT_SIT_TRANCADA"].fillna(0) +
        dfx["QT_SIT_DESVINCULADO"].fillna(0) +
        dfx["QT_SIT_TRANSFERIDO"].fillna(0) +
        dfx["QT_SIT_FALECIDO"].fillna(0)
    )
    dfx = dfx[dfx["QT_MAT"] > 0].copy()  # evita divisão por zero
    dfx["TAXA_EVASAO"] = dfx["QT_EVASAO_TOTAL"] / dfx["QT_MAT"]
    return dfx

df_train_raw = add_evasao_cols(df_train_raw)
df_val_raw   = add_evasao_cols(df_val_raw)

print("Anos treino:", sorted(df_train_raw["NU_ANO_CENSO"].unique().tolist()))
print("Anos validação:", sorted(df_val_raw["NU_ANO_CENSO"].unique().tolist()))

Anos treino: [2014, 2015, 2016, 2017, 2018, 2019, 2020]
Anos validação: [2021, 2022]


In [ ]:
#  Normalização (fit no TREINO; aplicar no VAL)
# Filtra CONT_COLS existentes
CONT_COLS = [c for c in CONT_COLS if c in df_train_raw.columns and c in df_val_raw.columns]

scaler = StandardScaler()
scaler.fit(df_train_raw[CONT_COLS])

def apply_scaling(dfx: pd.DataFrame) -> pd.DataFrame:
    dfx2 = dfx.copy()
    dfx2[CONT_COLS] = scaler.transform(dfx2[CONT_COLS])
    return dfx2

df_train = apply_scaling(df_train_raw)
df_val   = apply_scaling(df_val_raw)

In [ ]:
# Seleção final de FEATURES
FEATURES = CONT_COLS + CAT_COLS
# Garante que todas existam
FEATURES = [c for c in FEATURES if c in df_train.columns and c in df_val.columns]

In [ ]:
# Função para criar sequências (janelas)
from tqdm.auto import tqdm # Importar tqdm

def build_sequences(df_all_history: pd.DataFrame,
                    df_target_years: pd.DataFrame,
                    history_window: int,
                    forecast_h: int,
                    group_keys: list,
                    features: list):
    """
    Cria amostras (X, y) por janelas deslizantes, por série temporal (group_keys).
    - df_all_history: histórico disponível (para construir o passado da série)
    - df_target_years: subconjunto que define em quais anos o alvo deve cair
    - history_window: tamanho da janela de entradas (ex.: 3 anos)
    - forecast_h: horizonte de previsão (ex.: 1 ano à frente)
    - features: colunas de entrada (numéricas + categóricas já numerificadas)

    Retorna:
      X_list: [N, history_window, num_features]
      y_list: [N]
      meta_list: lista de dicts com 'target_year' e 'group_keys'
    """
    X_list, y_list, meta_list = [], [], []
    target_years_set = set(df_target_years["NU_ANO_CENSO"].unique().tolist())

    # Agrupamento por entidade da série
    # Adicionar tqdm para a barra de progresso
    grouped = df_all_history.groupby(group_keys)
    for keys, g in tqdm(grouped, desc="Building sequences"):
        g = g.sort_values("NU_ANO_CENSO")
        n = len(g)
        if n < history_window + forecast_h:
            continue

        for end_idx in range(history_window, n - forecast_h + 1):
            start_hist = end_idx - history_window
            # histórico (janela)
            x_hist = g.iloc[start_hist:end_idx][features].values.astype(np.float32)

            # alvo (no passo end_idx + forecast_h - 1)
            target_row = g.iloc[end_idx + (forecast_h - 1)]
            target_year = int(target_row["NU_ANO_CENSO"])
            if target_year not in target_years_set:
                continue

            y = float(target_row["TAXA_EVASAO"])

            X_list.append(x_hist)
            y_list.append(y)
            meta_list.append({
                "target_year": target_year,
                "group_keys": keys
            })

    return X_list, y_list, meta_list

In [ ]:
# Criar sequências para TREINO e VALIDAÇÃO
# Para TREINO: alvos dentro do período de treino, histórico = próprio treino
latest_year_train = df_train["NU_ANO_CENSO"].max()
# df_train_filtered = df_train[df_train["NU_ANO_CENSO"] >= latest_year_train - 3] # Últimos 4 anos (ano atual - 3 anos anteriores)


X_train, y_train, meta_train = build_sequences(
    df_all_history=df_train, # Usar dados filtrados
    df_target_years=df_train, # Alvos também nos anos filtrados
    history_window=HISTORY_WINDOW,
    forecast_h=FORECAST_H,
    group_keys=GROUP_KEYS,
    features=FEATURES
)

Building sequences:   0%|          | 0/38713 [00:00<?, ?it/s]

In [ ]:
print("Anos únicos no dataframe de treino:", sorted(df_train["NU_ANO_CENSO"].unique().tolist()))

In [11]:
# Para VALIDAÇÃO: alvos nos anos de validação; histórico (treino + validação)
df_hist_for_val = pd.concat([df_train, df_val], axis=0, ignore_index=True)
X_val, y_val, meta_val = build_sequences(
    df_all_history=df_hist_for_val,
    df_target_years=df_val,
    history_window=HISTORY_WINDOW,
    forecast_h=FORECAST_H,
    group_keys=GROUP_KEYS,
    features=FEATURES
)

Building sequences:   0%|          | 0/46175 [00:00<?, ?it/s]

In [12]:
print(f"Sequências geradas — Treino: {len(X_train)} | Validação: {len(X_val)}")
num_features = len(FEATURES)
seq_len = HISTORY_WINDOW
print("Num features:", num_features, "| seq_len:", seq_len)

Sequências geradas — Treino: 891701 | Validação: 800309
Num features: 16 | seq_len: 3


In [13]:
#  CONSTRUÇÃO DO MODELO
class LSTMEvasao(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)   # saída: taxa de evasão (regressão)
        )

    def forward(self, x):
        # x: [batch, seq_len, input_size]
        out, (h_n, c_n) = self.lstm(x)     # h_n: [num_layers, batch, hidden]
        h_last = h_n[-1]                   # [batch, hidden]
        y_hat = self.head(h_last).squeeze(-1)  # [batch]
        return y_hat

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMEvasao(input_size=num_features, hidden_size=64, num_layers=2, dropout=0.2).to(device)
print(model)

LSTMEvasao(
  (lstm): LSTM(16, 64, num_layers=2, batch_first=True, dropout=0.2)
  (head): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [ ]:
# Empacotar dados em DataLoader
def to_tensors(X_list, y_list):
    X = np.stack(X_list, axis=0)  # [N, seq_len, num_features]
    y = np.array(y_list, dtype=np.float32)  # [N]
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)
    return X_t, y_t

Xtr_t, ytr_t = to_tensors(X_train, y_train)
Xva_t, yva_t = to_tensors(X_val, y_val)

train_ds = TensorDataset(Xtr_t, ytr_t)
val_ds   = TensorDataset(Xva_t, yva_t)

train_dl = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=False)
val_dl   = DataLoader(val_ds, batch_size=256, shuffle=False, drop_last=False)

# Perda, otimizador, early stopping
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

best_val = float("inf")
patience = 5
wait = 0
best_state = None
EPOCHS = 30

for epoch in range(1, EPOCHS+1):
    # --- treino ---
    model.train()
    train_loss = 0.0
    # Adicionar tqdm para a barra de progresso no treino
    for xb, yb in tqdm(train_dl, desc=f"Epoch {epoch:02d} [Treino]"):
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_dl.dataset)

    # validação
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        # Adicionar tqdm para a barra de progresso na validação
        for xb, yb in tqdm(val_dl, desc=f"Epoch {epoch:02d} [Validação]"):
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(val_dl.dataset)

    print(f"Época {epoch:02d} | train MSE: {train_loss:.6f} | val MSE: {val_loss:.6f}")

    # early stopping
    if val_loss < best_val - 1e-6:
        best_val = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(" Early stopping acionado.")
            break

# Carregar melhor estado
if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)
    print(f" Melhor val MSE: {best_val:.6f}")

Epoch 01 [Treino]:   0%|          | 0/3484 [00:00<?, ?it/s]

Epoch 01 [Validação]:   0%|          | 0/3127 [00:00<?, ?it/s]

Época 01 | train MSE: 8.245005 | val MSE: 3.910335


Epoch 02 [Treino]:   0%|          | 0/3484 [00:00<?, ?it/s]

Epoch 02 [Validação]:   0%|          | 0/3127 [00:00<?, ?it/s]

...

Época 15 | train MSE: 8.240053 | val MSE: 3.910964


Epoch 16 [Treino]:   0%|          | 0/3484 [00:00<?, ?it/s]

Epoch 16 [Validação]:   0%|          | 0/3127 [00:00<?, ?it/s]

Época 16 | train MSE: 8.240098 | val MSE: 3.914455


Epoch 17 [Treino]:   0%|          | 0/3484 [00:00<?, ?it/s]

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_val = []
    for xb, yb in val_dl:
        xb = xb.to(device)
        preds = model(xb).cpu().numpy()
        y_pred_val.append(preds)
    y_pred_val = np.concatenate(y_pred_val, axis=0)

y_true_val = yva_t.numpy()

# --- Métricas ---
def mae(y, yhat): return float(np.mean(np.abs(y - yhat)))
def rmse(y, yhat): return float(sqrt(np.mean((y - yhat)**2)))
def r2(y, yhat):
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    return float(1 - ss_res/ss_tot) if ss_tot > 0 else float("nan")

print(f"Val MAE : {mae(y_true_val, y_pred_val):.6f}")
print(f"Val RMSE: {rmse(y_true_val, y_pred_val):.6f}")
print(f"Val R^2 : {r2(y_true_val, y_pred_val):.6f}")

In [ ]:
# --- Gráfico 1: Dispersão y_true vs y_pred ---
plt.figure(figsize=(6,6))
plt.scatter(y_true_val, y_pred_val, alpha=0.3, s=8)
plt.plot([0,1],[0,1])
plt.title("Validação: y verdadeiro vs y predito")
plt.xlabel("Taxa de evasão (real)")
plt.ylabel("Taxa de evasão (prevista)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# --- Gráfico 2: Curvas anuais agregadas por modalidade (0/1) ---
# Precisamos resgatar os metadados (anos alvo) e modalidades correspondentes às sequências de val.
# meta_val tem target_year e group_keys = (CO_IES, CO_CINE_ROTULO_id, Modalidade)
val_meta_df = pd.DataFrame(meta_val)
val_meta_df["y_true"] = y_true_val
val_meta_df["y_pred"] = y_pred_val

# extrair Modalidade do tuple group_keys (índice 2)
val_meta_df["Modalidade"] = val_meta_df["group_keys"].apply(lambda t: t[2])

# Agregar por ano e modalidade
agg_val = (
    val_meta_df
    .groupby(["target_year", "Modalidade"], as_index=False)
    .agg(y_true_mean=("y_true","mean"), y_pred_mean=("y_pred","mean"))
    .sort_values(["target_year","Modalidade"])
)

# Pivot para duas linhas por modalidade
pivot_true = agg_val.pivot(index="target_year", columns="Modalidade", values="y_true_mean")
pivot_pred = agg_val.pivot(index="target_year", columns="Modalidade", values="y_pred_mean")

plt.figure(figsize=(10,6))
for mod in [0,1]:
    if mod in pivot_true.columns:
        plt.plot(pivot_true.index, pivot_true[mod].values, marker="o", linewidth=2, label=f"Real - Modalidade {mod}")
    if mod in pivot_pred.columns:
        plt.plot(pivot_pred.index, pivot_pred[mod].values, marker="o", linewidth=2, linestyle="--", label=f"Previsto - Modalidade {mod}")

In [ ]:
plt.title("Validação: Taxa média de evasão por ano e modalidade")
plt.xlabel("Ano do Censo")
plt.ylabel("Taxa de evasão média")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()